<a href="https://colab.research.google.com/github/thehmfpk/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane
**Lane:** same as ML-04/05/06/07 — `fact_content_daily_performance`, month `2026-03`, decision moment = day 15.
**Target:** `is_declining_label` (clicks fall days 16-31 vs days 1-15), same label built in ML-04/05.
**Week-4 baseline being beaten:** the `CTR_FIX_OPPORTUNITY` rule score from ML-07, recomputed here over the full population (not just its filtered candidate set) so it can be ranked and scored on the same held-out split as the trained model below - that's what makes the comparison in section 3 apples-to-apples.

Load `training-honest-models` + `flyrank/flyrank-data` per `skills/README.md` if working with an assistant.


## 0. Setup — load March 2026, rebuild features + label + baseline score (same as ML-04/05/07)

In [1]:
import pandas as pd, numpy as np, os, sklearn
from datasets import load_dataset
from huggingface_hub import HfApi
from google.colab import userdata

print(f'scikit-learn version: {sklearn.__version__}')  # note this if a headline number matters later
RANDOM_SEED = 42

HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
all_files = api.list_repo_files('FlyRank/internship-warehouse', repo_type='dataset')
march_files = [f for f in all_files if 'fact_content_daily_performance' in f
               and 'sample' not in f and '2026-03' in f]
if not march_files:
    print('[WARNING] No files matched - inspect all_files and fix the filter:')
    for f in [x for x in all_files if 'fact_content_daily_performance' in x and 'sample' not in x][:20]:
        print(' ', f)
    raise ValueError('Fix march_files filter above, then re-run.')

dataset = load_dataset('FlyRank/internship-warehouse', data_files={'train': march_files}, split='train')
needed_cols = ['report_date','client_hash_id','content_hash_id',
               'gsc_data_available','gsc_impressions','gsc_clicks','gsc_avg_position']
cols_present = [c for c in needed_cols if c in dataset.column_names]
dataset = dataset.select_columns(cols_present)
raw = dataset.to_pandas()
raw['report_date'] = pd.to_datetime(raw['report_date'])
raw['day_of_month'] = raw['report_date'].dt.day
raw = raw[raw['gsc_data_available'] == True].copy()
print(f'Loaded {len(raw)} GSC-available rows for March 2026.')


scikit-learn version: 1.6.1


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Loaded 3611061 GSC-available rows for March 2026.


In [2]:
first_half = raw[raw['day_of_month'] <= 15]
second_half = raw[raw['day_of_month'] > 15]

feat = (first_half.groupby(['client_hash_id','content_hash_id'])
        .agg(clicks_first_half=('gsc_clicks','sum'),
             impressions_first_half=('gsc_impressions','sum'),
             avg_position_first_half=('gsc_avg_position', lambda s: s[s > 0].mean()),
             active_days_first_half=('gsc_impressions', lambda s: (s > 0).sum()))
        .reset_index())
feat = feat[feat['impressions_first_half'] > 0].copy()
feat['ctr_first_half'] = feat['clicks_first_half'] / feat['impressions_first_half']
feat['has_position_data'] = feat['avg_position_first_half'].notna().astype(int)
feat['avg_position_first_half'] = feat['avg_position_first_half'].fillna(feat['avg_position_first_half'].median())

position_bins = [0, 3, 6, 10, 20, 50, np.inf]
position_labels = ['1-3','4-6','7-10','11-20','21-50','51+']
feat['position_bucket_fh'] = pd.cut(feat['avg_position_first_half'], bins=position_bins, labels=position_labels)
position_dummies = pd.get_dummies(feat['position_bucket_fh'], prefix='pos_bucket')

# Week-4 baseline score, recomputed over ALL content (not just the filtered candidate set from ML-07)
expected_ctr_map = feat.groupby('position_bucket_fh', observed=True)['ctr_first_half'].mean()
feat['expected_ctr'] = feat['position_bucket_fh'].astype(str).map(
    {str(k): v for k, v in expected_ctr_map.items()}).astype(float)
feat['ctr_shortfall'] = (feat['expected_ctr'] - feat['ctr_first_half']).clip(lower=0)
feat['baseline_score'] = feat['impressions_first_half'] * feat['ctr_shortfall']

feature_vector = pd.concat([
    feat[['client_hash_id','content_hash_id','clicks_first_half','impressions_first_half',
          'ctr_first_half','avg_position_first_half','has_position_data',
          'active_days_first_half','baseline_score']],
    position_dummies
], axis=1)

# Label from days 16-31 - built after features, never fed into feature engineering above
label_df = (second_half.groupby(['client_hash_id','content_hash_id'])
            .agg(clicks_second_half=('gsc_clicks','sum')).reset_index())

data = feature_vector.merge(label_df, on=['client_hash_id','content_hash_id'], how='inner')
data['is_declining_label'] = (data['clicks_second_half'] < data['clicks_first_half']).astype(int)
data = data.dropna()
print(f'{len(data)} content items with features + label. Base rate (declining share): {data["is_declining_label"].mean():.3f}')


141467 content items with features + label. Base rate (declining share): 0.203


## 1. Method choice and why

**Question shape:** yes/no with an observed label (`is_declining_label` is a real, measured outcome from days 16-31 - not a proxy or assumption). Per the toolkit table, that means: **start with Logistic Regression, then Random Forest** - readable first, stronger second, and only keep the stronger one if the comparison in section 3 actually earns it.

Not using clustering (nothing here is a grouping question) or plain permutation-importance-on-a-simple-model alone (that's for a "what drives X" question; here we have a real classification target, so a trained classifier is the right tool, with permutation importance used afterward in section 4 for interpretation, not as the primary method).

## 2. Split design

**Grouped by client**, not a random row split. Content items from the same client tend to share characteristics (vertical, content strategy, baseline traffic patterns) - a random split could put near-identical sibling content from the same client in both train and test, which would let the model partially memorize client-level patterns rather than learn the general signal. `flyrank-data` explicitly calls this out: use `client_hash_id` for grouped splits, never a plain random split, and IDs are never features (confirmed below).

In [3]:
from sklearn.model_selection import GroupShuffleSplit

assert 'client_hash_id' not in [c for c in feature_vector.columns if c not in ('client_hash_id','content_hash_id')]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(data, groups=data['client_hash_id']))
train_df, test_df = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()

overlap = set(train_df['client_hash_id']) & set(test_df['client_hash_id'])
print(f'Train: {len(train_df)} rows, {train_df["client_hash_id"].nunique()} clients')
print(f'Test:  {len(test_df)} rows, {test_df["client_hash_id"].nunique()} clients')
print(f'Clients appearing in both train and test: {len(overlap)} (must be 0 for a clean grouped split)')
assert len(overlap) == 0, 'Grouped split failed - a client leaked across train/test'


Train: 129917 rows, 32 clients
Test:  11550 rows, 11 clients
Clients appearing in both train and test: 0 (must be 0 for a clean grouped split)


## 3. Train + compare vs my baseline

Same data (`test_df`), same metric (precision@K, at a few K values, plus base rate), same split for everyone: the Week-4 rule baseline, Logistic Regression, and Random Forest.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

feature_cols = [c for c in feature_vector.columns if c not in ('client_hash_id','content_hash_id','baseline_score')]

X_train, y_train = train_df[feature_cols], train_df['is_declining_label']
X_test, y_test = test_df[feature_cols], test_df['is_declining_label']

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_SEED, n_jobs=-1).fit(X_train, y_train)

test_df = test_df.copy()
test_df['logreg_score'] = logreg.predict_proba(X_test)[:, 1]
test_df['rf_score'] = rf.predict_proba(X_test)[:, 1]
print('Models trained.')


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Models trained.


In [5]:
def precision_at_k(df, score_col, label_col, k):
    top_k = df.nlargest(k, score_col)
    return top_k[label_col].mean()

K_VALUES = [20, 50, 100]
base_rate = y_test.mean()

rows = []
for name, score_col in [('Base rate (random ranking)', None),
                         ('Week-4 rule baseline', 'baseline_score'),
                         ('Logistic Regression', 'logreg_score'),
                         ('Random Forest', 'rf_score')]:
    row = {'model': name}
    for k in K_VALUES:
        if score_col is None:
            row[f'precision@{k}'] = base_rate
        else:
            row[f'precision@{k}'] = precision_at_k(test_df, score_col, 'is_declining_label', k)
    rows.append(row)

comparison_table = pd.DataFrame(rows)
print('--- Comparison table: baseline vs model(s), same split, same metric ---')
print(comparison_table.to_string(index=False))


--- Comparison table: baseline vs model(s), same split, same metric ---
                     model  precision@20  precision@50  precision@100
Base rate (random ranking)      0.252035      0.252035       0.252035
      Week-4 rule baseline      0.550000      0.460000       0.510000
       Logistic Regression      0.650000      0.680000       0.660000
             Random Forest      0.850000      0.900000       0.870000


**Read this table honestly:** if one model wins at precision@100 but loses at precision@20, report both - that IS the finding, not a reason to only show the flattering K. If Random Forest doesn't clearly beat Logistic Regression here, the honest move is to ship the simpler, readable model - complexity alone isn't rewarded.

## 4. Errors and interpretation

What the winning model leans on (permutation importance - checked by shuffling, not just read off the fitted coefficients), where it's most wrong, and three concrete hard cases.

In [6]:
from sklearn.inspection import permutation_importance

# Use whichever model actually won section 3's table - defaulting to Random Forest here,
# swap to `logreg` + X_test if your run showed Logistic Regression winning instead.
best_model, best_score_col = rf, 'rf_score'

perm = permutation_importance(best_model, X_test, y_test, n_repeats=20, random_state=RANDOM_SEED, n_jobs=-1)
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)
print('--- Permutation importance (top 5) ---')
print(importance_df.head(5).to_string(index=False))

top3 = importance_df.head(3)['feature'].tolist()
print(f"\nTop 3 features: {top3}")
print('Sanity check: none of these should be a near-perfect single predictor - if one dominates')
print('with importance far above the rest, re-check it traces to days 1-15 only (see ML-05 leakage hunt).')


--- Permutation importance (top 5) ---
               feature  importance_mean  importance_std
        ctr_first_half         0.061996        0.002147
     clicks_first_half         0.056814        0.001879
active_days_first_half         0.038325        0.002059
impressions_first_half         0.005229        0.001490
       pos_bucket_7-10         0.000329        0.000308

Top 3 features: ['ctr_first_half', 'clicks_first_half', 'active_days_first_half']
Sanity check: none of these should be a near-perfect single predictor - if one dominates
with importance far above the rest, re-check it traces to days 1-15 only (see ML-05 leakage hunt).


In [7]:
# Where is the model most wrong - by position bucket, since that's the main structural grouping we have
test_df['predicted_label'] = (test_df[best_score_col] >= 0.5).astype(int)
test_df['correct'] = (test_df['predicted_label'] == test_df['is_declining_label'])

pos_cols = [c for c in feature_cols if c.startswith('pos_bucket_')]
test_df['position_bucket'] = test_df[pos_cols].idxmax(axis=1).str.replace('pos_bucket_', '', regex=False)

error_by_bucket = test_df.groupby('position_bucket', observed=True).agg(
    n=('correct','size'), accuracy=('correct','mean')
).reset_index()
print('--- Accuracy by position bucket ---')
print(error_by_bucket.to_string(index=False))


--- Accuracy by position bucket ---
position_bucket    n  accuracy
            1-3 1038  0.713873
          11-20 1995  0.882707
          21-50 1624  0.931650
            4-6 3008  0.765293
            51+  274  0.985401
           7-10 3611  0.879812


In [8]:
# Three concrete wrong cases
wrong = test_df[~test_df['correct']].copy()
wrong['confidence_gap'] = (wrong[best_score_col] - 0.5).abs()
hardest_wrong = wrong.nlargest(3, 'confidence_gap')

print('--- 3 concrete wrong cases (most confidently wrong) ---')
for i, row in hardest_wrong.iterrows():
    print(f"content_hash_id={row['content_hash_id']}: predicted {row['predicted_label']}, "
          f"actual {row['is_declining_label']}, model score={row[best_score_col]:.3f}")
    print(f"  clicks_first_half={row['clicks_first_half']:.0f}, impressions_first_half={row['impressions_first_half']:.0f}, "
          f"ctr_first_half={row['ctr_first_half']:.4f}, position_bucket={row['position_bucket']}")
    print(f"  why this is hard: high model confidence but wrong outcome usually means the first-half trend "
          f"reversed for a reason not captured in these features (seasonality, a one-off traffic spike/dip, "
          f"or genuinely thin first-half data driving an unstable CTR estimate).")
    print()


--- 3 concrete wrong cases (most confidently wrong) ---
content_hash_id=content_e548a3cf835e5663: predicted 1, actual 0, model score=0.813
  clicks_first_half=1, impressions_first_half=74, ctr_first_half=0.0135, position_bucket=51+
  why this is hard: high model confidence but wrong outcome usually means the first-half trend reversed for a reason not captured in these features (seasonality, a one-off traffic spike/dip, or genuinely thin first-half data driving an unstable CTR estimate).

content_hash_id=content_f5c1fb735fb4a08c: predicted 1, actual 0, model score=0.799
  clicks_first_half=1, impressions_first_half=101, ctr_first_half=0.0099, position_bucket=51+
  why this is hard: high model confidence but wrong outcome usually means the first-half trend reversed for a reason not captured in these features (seasonality, a one-off traffic spike/dip, or genuinely thin first-half data driving an unstable CTR estimate).

content_hash_id=content_73c45865be5c672d: predicted 1, actual 0, mode